<a href="https://colab.research.google.com/github/Marina4ij/dz3/blob/main/dz3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd
import os
import glob

# 1. Настройки
DB_NAME = "cicids2017.db"
CSV_FOLDER = "./CICIDS2017_CSVs" # Путь к папке, куда вы распаковали CSV файлы

# Создаем подключение
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# 2. Создание схемы базы данных
print("Создание таблиц...")

# Справочник типов атак (Labels)
cursor.execute('''
CREATE TABLE IF NOT EXISTS attack_labels (
    label_id INTEGER PRIMARY KEY AUTOINCREMENT,
    label_name TEXT UNIQUE NOT NULL,
    category TEXT NOT NULL, -- Например: 'Benign', 'DDoS', 'Web Attack', 'Bot'
    is_malicious BOOLEAN NOT NULL
)
''')

# Основная таблица сетевых потоков (Network Flows)
# Мы создадим её динамически на основе колонок из CSV,
# но добавим внешние ключи для связей.
cursor.execute('''
CREATE TABLE IF NOT EXISTS network_flows (
    flow_id TEXT PRIMARY KEY,
    timestamp TEXT,
    src_ip TEXT,
    src_port INTEGER,
    dst_ip TEXT,
    dst_port INTEGER,
    protocol TEXT,
    flow_duration REAL,
    total_fwd_packets INTEGER,
    total_bwd_packets INTEGER,
    label_id INTEGER,
    FOREIGN KEY (label_id) REFERENCES attack_labels(label_id)
)
''')
# Примечание: в реальном проекте таблица network_flows будет содержать все ~80 колонок.
# Pandas создаст недостающие колонки автоматически при первом же insert.

conn.commit()

# 3. Заполнение справочника атак (с учетом проблем кодировки в оригинальном датасете)
print("Заполнение справочника атак...")
labels_data = [
    ('BENIGN', 'Benign', False),
    ('DDoS', 'DDoS', True),
    ('PortScan', 'Reconnaissance', True),
    ('Bot', 'Botnet', True),
    ('Infiltration', 'Infiltration', True),
    ('Web Attack \x96 Brute Force', 'Web Attack', True), # Специфичный символ из оригинала
    ('Web Attack \x96 XSS', 'Web Attack', True),
    ('Web Attack \x96 Sql Injection', 'Web Attack', True),
    ('FTP-Patator', 'Brute Force', True),
    ('SSH-Patator', 'Brute Force', True),
    ('DoS slowloris', 'DoS', True),
    ('DoS Slowhttptest', 'DoS', True),
    ('DoS Hulk', 'DoS', True),
    ('DoS GoldenEye', 'DoS', True),
    ('Heartbleed', 'DoS', True)
]

# Очищаем названия от возможных артефактов кодировки перед вставкой
clean_labels = []
for name, cat, is_mal in labels_data:
    clean_name = name.replace('\x96', '-').replace('\xef\xbf\xbd', '-').strip()
    clean_labels.append((clean_name, cat, is_mal))

cursor.executemany('''
    INSERT OR IGNORE INTO attack_labels (label_name, category, is_malicious)
    VALUES (?, ?, ?)
''', clean_labels)
conn.commit()

# Создаем словарь для быстрого маппинга label_name -> label_id
cursor.execute("SELECT label_name, label_id FROM attack_labels")
label_mapping = {row[0]: row[1] for row in cursor.fetchall()}
# Добавляем варианты с разными тире на случай, если парсинг прошел иначе
label_mapping['Web Attack - Brute Force'] = label_mapping.get('Web Attack - Brute Force', 6)

# 4. Функция для очистки и загрузки CSV (Chunking для экономии RAM)
def load_cicids_csv(file_path):
    print(f"Обработка {os.path.basename(file_path)}...")

    # Читаем частями (chunk), чтобы не переполнить оперативную память
    chunk_size = 100000
    total_rows = 0

    for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
        # Очистка данных (критично для CICIDS2017!)
        # 1. Заменяем бесконечности и некорректные значения на NaN
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.fillna(0, inplace=True)

        # 2. Чистим названия атак от битых символов (артефакты кодировки)
        chunk['Label'] = chunk['Label'].apply(lambda x: str(x).replace('\x96', '-').replace('\xef\xbf\xbd', '-').strip())

        # 3. Маппим текстовые лейблы в ID из нашей БД
        chunk['label_id'] = chunk['Label'].map(label_mapping).fillna(1) # 1 - это BENIGN

        # 4. Оставляем только нужные колонки (или все, если хотите хранить 80 фич)
        # Для примера возьмем базовые + label_id. Если нужно все 80, просто уберите этот drop.
        cols_to_keep = [
            'Flow ID', 'Timestamp', 'Source IP', 'Source Port',
            'Destination IP', 'Destination Port', 'Protocol', 'Flow Duration',
            'Total Fwd Packets', 'Total Backward Packets', 'Label', 'label_id'
        ]

        # Фильтруем только существующие колонки (на случай опечаток в заголовках CSV)
        cols_to_keep = [c for c in cols_to_keep if c in chunk.columns]
        db_chunk = chunk[cols_to_keep]

        # Переименовываем колонки для соответствия БД (без пробелов)
        db_chunk.columns = [
            'flow_id', 'timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port',
            'protocol', 'flow_duration', 'total_fwd_packets', 'total_bwd_packets',
            'label_name', 'label_id'
        ]

        # Загружаем в SQLite
        db_chunk.to_sql('network_flows', conn, if_exists='append', index=False, method='multi')
        total_rows += len(db_chunk)

    print(f"  -> Загружено строк: {total_rows}")

# 5. Запуск загрузки всех CSV из папки
import numpy as np # нужен для np.inf

if os.path.exists(CSV_FOLDER):
    csv_files = glob.glob(os.path.join(CSV_FOLDER, "*.csv"))
    if not csv_files:
        print(f"В папке {CSV_FOLDER} не найдено CSV файлов.")
    else:
        for file in csv_files:
            load_cicids_csv(file)
else:
    print(f"Папка {CSV_FOLDER} не найдена. Создайте её и скачайте туда CSV файлы CICIDS2017.")

# 6. Проверка и закрытие
print("\nСтатистика загрузки:")
cursor.execute("SELECT COUNT(*) FROM network_flows")
print(f"Всего сетевых потоков: {cursor.fetchone()[0]:,}")

cursor.execute('''
    SELECT al.label_name, al.category, COUNT(nf.flow_id) as flow_count
    FROM network_flows nf
    JOIN attack_labels al ON nf.label_id = al.label_id
    GROUP BY al.label_name
    ORDER BY flow_count DESC
''')
print("\nРаспределение трафика по типам:")
for row in cursor.fetchall():
    print(f"  {row[0]:<25} ({row[1]:<15}): {row[2]:>10,} потоков")

conn.close()
print(f"\n✓ База данных '{DB_NAME}' успешно создана и заполнена!")

Создание таблиц...
Заполнение справочника атак...
Папка ./CICIDS2017_CSVs не найдена. Создайте её и скачайте туда CSV файлы CICIDS2017.

Статистика загрузки:
Всего сетевых потоков: 0

Распределение трафика по типам:

✓ База данных 'cicids2017.db' успешно создана и заполнена!


CICIDS2017 / CSE-CIC-IDS2018 — стандартные датасеты от Канадского института кибербезопасности. Содержат нормальный сетевой трафик и множество видов атак (DDoS, брутфорс, SQL-инъекции и т.д.).
Где найти: Сайт UNB (Canadian Institute for Cybersecurity) или Kaggle.
CICIDS2017: https://www.unb.ca/cic/datasets/ids-2017.html (CSV-файлы с сетевым трафиком и атаками)

Важные нюансы при работе с CICIDS2017
Проблема Infinity и NaN: В оригинальных CSV-файлах CICIDS2017 в некоторых числовых колонках (например, Flow Bytes/s) встречаются значения Infinity. Если попытаться загрузить их в SQLite или Pandas без очистки, скрипт упадет. В коде выше это решено строкой: chunk.replace([np.inf, -np.inf], np.nan, inplace=True).
Кодировка названий атак: Из-за того, что датасет собирался в Windows, в названиях атак вроде Web Attack – Brute Force вместо обычного тире стоит битый символ (часто \x96 или \xef\xbf\xbd). В скрипте добавлена очистка str(x).replace('\x96', '-').
Огромный размер: Файлы весят больше 2 ГБ. Загрузка в SQLite может занять от 10 до 40 минут в зависимости от вашего SSD. Использование chunksize=100000 гарантирует, что у вас не закончится оперативная память (OOM).
Все 80 колонок: В примере я оставил только ~12 самых важных колонок для наглядности. Если вам нужны все 80 признаков для обучения модели, просто закомментируйте блок cols_to_keep = ... и db_chunk = chunk[cols_to_keep]. Pandas автоматически создаст в SQLite все недостающие колонки при первом же to_sql.

Спроектируем БД по принципу "звезда": справочники (измерения) + основная таблица фактов + агрегатная таблица.
sql
-- Справочник типов атак
CREATE TABLE attack_types (
    attack_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL,           -- Название атаки
    category TEXT NOT NULL,              -- Категория (DDoS, Web Attack, Bot...)
    severity TEXT CHECK(severity IN ('Low', 'Medium', 'High', 'Critical')),
    description TEXT,                    -- Описание
    is_malicious BOOLEAN NOT NULL        -- 0 = BENIGN, 1 = атака
);

-- Справочник сетевых протоколов
CREATE TABLE protocols (
    protocol_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL,           -- TCP, UDP, ICMP
    protocol_number INTEGER NOT NULL     -- 6 = TCP, 17 = UDP, 1 = ICMP
);

-- Основная таблица сетевых потоков (факты)
CREATE TABLE network_flows (
    flow_id TEXT PRIMARY KEY,            -- Уникальный ID потока
    flow_timestamp DATETIME NOT NULL,    -- Временная метка начала потока
    flow_date DATE NOT NULL,             -- Дата (для агрегации по дням)
    src_ip TEXT NOT NULL,                -- IP-источник
    src_port INTEGER NOT NULL,           -- Порт источника (0-65535)
    dst_ip TEXT NOT NULL,                -- IP-назначения
    dst_port INTEGER NOT NULL,           -- Порт назначения
    protocol_id INTEGER NOT NULL,        -- FK к protocols
    flow_duration REAL,                  -- Длительность в микросекундах
    total_fwd_packets INTEGER,           -- Пакетов в прямом направлении
    total_bwd_packets INTEGER,           -- Пакетов в обратном
    total_length_fwd INTEGER,            -- Байт в прямом направлении
    total_length_bwd INTEGER,            -- Байт в обратном
    label TEXT NOT NULL,                 -- Оригинальная метка из CSV
    attack_id INTEGER NOT NULL,          -- FK к attack_types
    is_malicious BOOLEAN NOT NULL,       -- Быстрая проверка: атака или нет
    FOREIGN KEY (protocol_id) REFERENCES protocols(protocol_id),
    FOREIGN KEY (attack_id) REFERENCES attack_types(attack_id)
);

-- Агрегатная таблица: статистика трафика по дням
CREATE TABLE daily_traffic_summary (
    summary_date DATE PRIMARY KEY,       -- Дата
    total_flows INTEGER,                 -- Всего потоков
    benign_flows INTEGER,                -- Легитимных
    malicious_flows INTEGER,             -- Атак
    total_bytes BIGINT,                  -- Суммарный трафик в байтах
    top_attack_category TEXT,            -- Самая частая категория атак
    avg_flow_duration REAL,              -- Средняя длительность потока
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

-- Индексы для ускорения запросов
CREATE INDEX idx_flows_date ON network_flows(flow_date);
CREATE INDEX idx_flows_attack ON network_flows(attack_id);
CREATE INDEX idx_flows_src_ip ON network_flows(src_ip);
CREATE INDEX idx_flows_dst_ip ON network_flows(dst_ip);
CREATE INDEX idx_flows_malicious ON network_flows(is_malicious);

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime

# === НАСТРОЙКИ ===
DB_NAME = "cicids2017_normalized.db"
CSV_FOLDER = "./CICIDS2017_CSVs"
CHUNK_SIZE = 100_000  # Размер чанка для экономии RAM

# === 1. СОЗДАНИЕ БД И СХЕМЫ ===
conn = sqlite3.connect(DB_NAME)
conn.execute("PRAGMA journal_mode=WAL")  # Ускоряет запись
conn.execute("PRAGMA synchronous=NORMAL")
cursor = conn.cursor()

# Выполняем SQL-схему из пункта 1
with open("schema.sql", "r") as f:
    schema_sql = f.read()
cursor.executescript(schema_sql)
conn.commit()

print("✓ Схема БД создана")

# === 2. ЗАПОЛНЕНИЕ СПРАВОЧНИКОВ ===
# Протоколы
protocols_data = [
    ('TCP', 6), ('UDP', 17), ('ICMP', 1), ('ARP', 2054), ('Other', 0)
]
cursor.executemany(
    'INSERT OR IGNORE INTO protocols (name, protocol_number) VALUES (?, ?)',
    protocols_data
)

# Типы атак (с очисткой битых символов)
attacks_data = [
    ('BENIGN', 'Benign', 'Low', 'Normal network traffic', 0),
    ('DDoS', 'DDoS', 'Critical', 'Distributed Denial of Service', 1),
    ('PortScan', 'Reconnaissance', 'Medium', 'Network port scanning', 1),
    ('Bot', 'Botnet', 'High', 'Botnet activity', 1),
    ('Infiltration', 'Infiltration', 'Critical', 'Network infiltration', 1),
    ('Web Attack - Brute Force', 'Web Attack', 'High', 'Brute force login attempts', 1),
    ('Web Attack - XSS', 'Web Attack', 'High', 'Cross-site scripting', 1),
    ('Web Attack - Sql Injection', 'Web Attack', 'Critical', 'SQL injection attack', 1),
    ('FTP-Patator', 'Brute Force', 'Medium', 'FTP brute force', 1),
    ('SSH-Patator', 'Brute Force', 'Medium', 'SSH brute force', 1),
    ('DoS slowloris', 'DoS', 'High', 'Slowloris DoS attack', 1),
    ('DoS Slowhttptest', 'DoS', 'High', 'Slow HTTP DoS', 1),
    ('DoS Hulk', 'DoS', 'High', 'HULK DoS attack', 1),
    ('DoS GoldenEye', 'DoS', 'High', 'GoldenEye DoS', 1),
    ('Heartbleed', 'DoS', 'Critical', 'OpenSSL Heartbleed exploit', 1)
]

cursor.executemany('''
    INSERT OR IGNORE INTO attack_types
    (name, category, severity, description, is_malicious)
    VALUES (?, ?, ?, ?, ?)
''', attacks_data)
conn.commit()

# Словари для быстрого маппинга
cursor.execute("SELECT protocol_number, protocol_id FROM protocols")
protocol_map = {row[0]: row[1] for row in cursor.fetchall()}
protocol_map['TCP'] = protocol_map[6]
protocol_map['UDP'] = protocol_map[17]
protocol_map['ICMP'] = protocol_map[1]

cursor.execute("SELECT name, attack_id, is_malicious FROM attack_types")
attack_rows = cursor.fetchall()
attack_map = {row[0]: row[1] for row in attack_rows}
malicious_map = {row[0]: row[2] for row in attack_rows}

print("Справочники заполнены")

# === 3. ФУНКЦИЯ ИМПОРТА CSV ===
def import_csv_to_db(file_path):
    """Импортирует один CSV-файл CICIDS2017 в БД чанками."""
    filename = os.path.basename(file_path)
    print(f"\n Обработка: {filename}")

    total_rows = 0
    for chunk_num, chunk in enumerate(pd.read_csv(
        file_path, chunksize=CHUNK_SIZE, low_memory=False
    ), 1):
        # Очистка: убираем inf, NaN
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.fillna(0, inplace=True)

        # Очистка названий атак от битой кодировки
        chunk['Label'] = chunk['Label'].astype(str).apply(
            lambda x: x.replace('\x96', '-').replace('\xef\xbf\xbd', '-').strip()
        )

        # Парсим timestamp -> DATE и DATETIME
        chunk['flow_timestamp'] = pd.to_datetime(
            chunk['Timestamp'], format='%d/%m/%Y %H:%M:%S', errors='coerce'
        )
        chunk['flow_date'] = chunk['flow_timestamp'].dt.date

        # Маппим протоколы
        if 'Protocol' in chunk.columns:
            chunk['protocol_id'] = chunk['Protocol'].map(
                lambda x: protocol_map.get(x, protocol_map.get(0, 5))
            )
        else:
            chunk['protocol_id'] = 5  # Other

        # Маппим атаки
        chunk['attack_id'] = chunk['Label'].map(
            lambda x: attack_map.get(x, 1)  # 1 = BENIGN по умолчанию
        )
        chunk['is_malicious'] = chunk['attack_id'].map(
            lambda x: malicious_map.get(x, 0)
        )

        # Формируем flow_id, если его нет
        if 'Flow ID' not in chunk.columns:
            chunk['flow_id'] = [f"{filename}_{chunk_num}_{i}"
                               for i in range(len(chunk))]

        # Отбираем нужные колонки (переименовываем под схему БД)
        cols_mapping = {
            'Flow ID': 'flow_id',
            'Source IP': 'src_ip',
            'Source Port': 'src_port',
            'Destination IP': 'dst_ip',
            'Destination Port': 'dst_port',
            'Flow Duration': 'flow_duration',
            'Total Fwd Packets': 'total_fwd_packets',
            'Total Backward Packets': 'total_bwd_packets',
            'Total Length of Fwd Packets': 'total_length_fwd',
            'Total Length of Bwd Packets': 'total_length_bwd',
        }

        # Берем только существующие колонки
        available_cols = {csv_col: db_col for csv_col, db_col in cols_mapping.items()
                         if csv_col in chunk.columns}

        db_chunk = chunk[list(available_cols.keys()) +
                        ['flow_timestamp', 'flow_date', 'protocol_id',
                         'Label', 'attack_id', 'is_malicious']].copy()
        db_chunk.rename(columns=available_cols, inplace=True)

        # Приводим типы для совместимости с SQLite
        db_chunk['src_port'] = db_chunk['src_port'].astype('Int64')
        db_chunk['dst_port'] = db_chunk['dst_port'].astype('Int64')
        db_chunk['flow_date'] = db_chunk['flow_date'].astype(str)
        db_chunk['flow_timestamp'] = db_chunk['flow_timestamp'].astype(str)

        # Вставляем в БД
        db_chunk.to_sql('network_flows', conn, if_exists='append',
                       index=False, method='multi')
        total_rows += len(db_chunk)
        print(f"  ├─ Чанк {chunk_num}: +{len(db_chunk):,} строк")

    print(f"  └─ Итого загружено: {total_rows:,} строк")
    return total_rows

# === 4. ЗАПУСК ИМПОРТА ВСЕХ CSV ===
if not os.path.exists(CSV_FOLDER):
    print(f"Папка {CSV_FOLDER} не найдена!")
else:
    csv_files = sorted(glob.glob(os.path.join(CSV_FOLDER, "*.csv")))
    if not csv_files:
        print(f"В папке нет CSV файлов")
    else:
        print(f"\n Найдено {len(csv_files)} файлов для импорта")
        grand_total = 0
        for csv_file in csv_files:
            grand_total += import_csv_to_db(csv_file)
        print(f"\n Всего импортировано: {grand_total:,} записей")

# === 5. ЗАПОЛНЕНИЕ АГРЕГАТНОЙ ТАБЛИЦЫ ===
print("\n Формирование daily_traffic_summary...")
cursor.execute('''
    INSERT OR REPLACE INTO daily_traffic_summary
    (summary_date, total_flows, benign_flows, malicious_flows,
     total_bytes, avg_flow_duration, top_attack_category)
    SELECT
        flow_date,
        COUNT(*) as total_flows,
        SUM(CASE WHEN is_malicious = 0 THEN 1 ELSE 0 END) as benign_flows,
        SUM(is_malicious) as malicious_flows,
        SUM(COALESCE(total_length_fwd, 0) + COALESCE(total_length_bwd, 0)) as total_bytes,
        AVG(flow_duration) as avg_flow_duration,
        (SELECT at2.category
         FROM network_flows nf2
         JOIN attack_types at2 ON nf2.attack_id = at2.attack_id
         WHERE nf2.flow_date = nf.flow_date AND nf2.is_malicious = 1
         GROUP BY at2.category
         ORDER BY COUNT(*) DESC LIMIT 1
        ) as top_attack_category
    FROM network_flows nf
    GROUP BY flow_date
''')
conn.commit()
print(f"✓ Добавлено {cursor.rowcount} дней в агрегатную таблицу")

# === 6. СТАТИСТИКА ===
print("\n" + "="*60)
print("СТАТИСТИКА БАЗЫ ДАННЫХ")
print("="*60)

cursor.execute("SELECT COUNT(*) FROM network_flows")
print(f"Всего потоков: {cursor.fetchone()[0]:>12,}")

cursor.execute("SELECT COUNT(*) FROM attack_types")
print(f"Типов атак:    {cursor.fetchone()[0]:>12,}")

cursor.execute("SELECT COUNT(DISTINCT flow_date) FROM network_flows")
print(f"Дней в данных: {cursor.fetchone()[0]:>12,}")

cursor.execute("SELECT SUM(total_length_fwd) + SUM(total_length_bwd) FROM network_flows")
total_bytes = cursor.fetchone()[0] or 0
print(f"Общий трафик:  {total_bytes/1e9:>12.2f} ГБ")

print("\n Топ-5 атак по количеству:")
cursor.execute('''
    SELECT at.name, at.category, COUNT(*) as cnt
    FROM network_flows nf
    JOIN attack_types at ON nf.attack_id = at.attack_id
    WHERE nf.is_malicious = 1
    GROUP BY at.attack_id
    ORDER BY cnt DESC
    LIMIT 5
''')
for i, row in enumerate(cursor.fetchall(), 1):
    print(f"  {i}. {row[0]:<30} ({row[1]:<15}) - {row[2]:>10,}")

conn.close()
print(f"\n База сохранена: {DB_NAME}")

импорт данных

аналитические запросы sql
-- 1. Распределение трафика по дням (проверка типа DATE)
SELECT flow_date,
       COUNT(*) as flows,
       SUM(is_malicious) as attacks
FROM network_flows
GROUP BY flow_date
ORDER BY flow_date;

-- 2. Топ IP-адресов-источников атак (проверка типа TEXT)
SELECT src_ip, COUNT(*) as attack_count
FROM network_flows
WHERE is_malicious = 1
GROUP BY src_ip
ORDER BY attack_count DESC
LIMIT 10;

-- 3. Средняя длительность атак по категориям (проверка типа REAL)
SELECT at.category,
       AVG(nf.flow_duration) as avg_duration,
       MAX(nf.flow_duration) as max_duration
FROM network_flows nf
JOIN attack_types at ON nf.attack_id = at.attack_id
WHERE nf.is_malicious = 1
GROUP BY at.category;

-- 4. Динамика атак по времени (проверка типа DATETIME)
SELECT strftime('%Y-%m-%d %H:00:00', flow_timestamp) as hour,
       COUNT(*) as flows,
       SUM(is_malicious) as attacks
FROM network_flows
WHERE flow_date = '2017-07-04'  -- Пример даты
GROUP BY hour
ORDER BY hour;

-- 5. Готовая сводка из агрегатной таблицы
SELECT * FROM daily_traffic_summary ORDER BY summary_date;